# 🏦 Pandas for auditors — Solutions: Lists & PDF Extraction

> ⚠️ This file contains the **solutions**. Try first with `exercice_pdf_listes.ipynb`!

## 0. PDF generation

In [ ]:
import fitz, re, pandas as pd

page_texts = [
    [
        "AML/CFT Surveillance Report — Q1 2024",
        "Compliance Department | Issued on 02/04/2024 | Period : 01/01/2024 – 31/03/2024",
        "",
        "── Incoming flows ──",
        "  A-001  Received on 08/01/2024  Origin : France         EUR  14 200",
        "  A-002  Received on 15/01/2024  Origin : Germany        EUR   6 300",
        "  A-003  Received on 03/02/2024  Origin : Panama         USD  48 000  ⚠ alert",
        "  A-004  Received on 20/02/2024  Origin : Luxembourg     EUR   9 750",
        "  A-005  Received on 11/03/2024  Origin : France         EUR  22 500",
        "  A-006  Received on 25/03/2024  Origin : Singapore      USD  17 800  ⚠ alert",
        "",
        "── Outgoing flows ──",
        "  B-001  Sent on 14/01/2024  Destination : Cyprus     EUR   9 900  ⚠ alert",
        "  B-002  Sent on 22/01/2024  Destination : Belgium    EUR   4 100",
        "  B-003  Sent on 07/02/2024  Destination : Malta      EUR   8 500  ⚠ alert",
        "  B-004  Sent on 28/02/2024  Destination : France     EUR  11 000",
    ],
    [
        "AML/CFT Surveillance Report — Q1 2024  (continued)",
        "",
        "── Outgoing flows (continued) ──",
        "  B-005  Sent on 05/03/2024  Destination : United Arab Emirates  USD  33 000  ⚠ alert",
        "  B-006  Sent on 18/03/2024  Destination : Switzerland CHF   5 600",
        "  B-007  Sent on 29/03/2024  Destination : Germany   EUR   2 900",
        "",
        "── Summary ──",
        "  Total incoming  : 118 550 EUR/USD",
        "  Total outgoing  :  75 000 EUR/USD/CHF",
        "  Flagged operations :       6",
        "",
        "Next review : 30/06/2024",
        "Compliance officer : Mr. Durand — Approved on 04/04/2024",
    ]
]

doc = fitz.open()
for content in page_texts:
    page = doc.new_page()
    page.insert_text((55, 60), "\n".join(content), fontsize=10.5)
doc.save("surveillance_report_q1_2024.pdf")
doc.close()
print("PDF generated: surveillance_report_q1_2024.pdf (2 pages)")

---
## Exercise 1 — List manipulation

In [ ]:
observed_countries = [
    'France', 'Germany', 'Panama', 'Luxembourg', 'Singapore',
    'Cyprus', 'Belgium', 'Malta', 'United Arab Emirates', 'Switzerland',
    'France', 'Luxembourg', 'Panama',
]

In [ ]:
# 1a. Length, first and last element
print("Number of elements:", len(observed_countries))
print("First:", observed_countries[0])
print("Last :", observed_countries[-1])

In [ ]:
# 1b. Remove duplicates and sort
unique_countries = sorted(list(set(observed_countries)))
print("Without duplicates, sorted:", unique_countries)

In [ ]:
# 1c. Comprehensions
short_countries = [p for p in unique_countries if len(p) <= 5]
countries_upper = [p.upper() for p in unique_countries]
print("Short countries (≤ 5 chars):", short_countries)
print("Uppercase                 :", countries_upper)

In [ ]:
# 1d. Filter the risk countries
risk_countries = ['Panama', 'Cyprus', 'Malta', 'Singapore', 'United Arab Emirates', 'Cayman Islands']
risk_countries_observed = [p for p in unique_countries if p in risk_countries]
print("Risk countries present in the data:", risk_countries_observed)

---
## Exercise 2 — Reading the PDF

In [ ]:
# 2a. Open, count pages, close
doc = fitz.open("surveillance_report_q1_2024.pdf")
print("Number of pages:", doc.page_count)
doc.close()

In [ ]:
# 2b. Extract and concatenate the text from all pages
doc = fitz.open("surveillance_report_q1_2024.pdf")
full_text = ""
for page in doc:
    full_text += page.get_text()
doc.close()

print("Preview (first 200 characters):")
print(full_text[:200])

---
## Exercise 3 — Extracting dates

In [ ]:
# 3a. Extract all DD/MM/YYYY dates
date_strings = re.findall(r'\d{2}/\d{2}/\d{4}', full_text)
print(f"Dates found ({len(date_strings)}):", date_strings)

In [ ]:
# 3b. Conversion to pandas datetime
dates_pd = pd.to_datetime(date_strings, format='%d/%m/%Y')
print("Earliest date:", dates_pd.min().date())
print("Most recent date :", dates_pd.max().date())

In [ ]:
# 3c. Bonus: dates in March (month == 3)
march_dates = [d for d in dates_pd if d.month == 3]
print("Dates in March:", len(march_dates), "→", [str(d.date()) for d in march_dates])

---
## Exercise 4 — Extracting countries

In [ ]:
reference_countries = [
    'France', 'Germany', 'Belgium', 'Luxembourg', 'Switzerland', 'Monaco',
    'Panama', 'Cyprus', 'Malta', 'Singapore', 'United Arab Emirates',
    'Cayman Islands', 'Bahamas', 'Jersey',
]

# 4a. Reference countries present in the PDF
countries_in_pdf = [p for p in reference_countries if p in full_text]
print("Countries in the PDF:", countries_in_pdf)

In [ ]:
# 4b. Intersection with risk_countries
flagged_countries = [p for p in countries_in_pdf if p in risk_countries]
print("Risk countries in the PDF:", flagged_countries)

---
## Exercise 5 — Building a DataFrame from the PDF

In [ ]:
# 5a. Extraction with regex
flow_pattern = r'([AB]-\d{3})\s+(Received|Sent) on (\d{2}/\d{2}/\d{4})\s+(?:Origin|Destination)\s*:\s*([\w\s]+?)\s+(EUR|USD|CHF)\s+([\d ]+)'
rows = re.findall(flow_pattern, full_text)
print(f"{len(rows)} rows extracted:")
for l in rows:
    print(" ", l)

In [ ]:
# 5b. Build the DataFrame
df_flows = pd.DataFrame(rows, columns=['ref', 'direction', 'date', 'country', 'currency', 'raw_amount'])
df_flows['date']   = pd.to_datetime(df_flows['date'], format='%d/%m/%Y')
df_flows['amount'] = df_flows['raw_amount'].str.replace(' ', '').astype(float)
df_flows = df_flows.drop(columns='raw_amount')
df_flows

In [ ]:
# 5c. alert column + export
df_flows['alert'] = df_flows['country'].isin(risk_countries)
print("Flagged flows:", df_flows['alert'].sum())
print()
print(df_flows[df_flows['alert']][['ref', 'date', 'direction', 'country', 'amount', 'currency']])

df_flows.to_excel('flows_surveillance_q1.xlsx', index=False)
print("\nFile flows_surveillance_q1.xlsx exported.")